In [ ]:
from pprint import pprint

import torch
from torch.utils.data import DataLoader

from src.config import DataConfig, DiffusionConfig, TrainConfig, VAEConfig
from src.data import load_ihdp
from src.model import DiffPO, _DiffusionBase
from train import evaluate

In [ ]:
VAE_CFG = VAEConfig(
    feature_dim=25,
    latent_dim=20,
    hidden_dim=64,
    encoder_num_layers=2,
    decoder_num_layers=1,
    aux_num_layers=1,
    a_decoder_hidden_dim=10,
)
TRAIN_CFG = TrainConfig(
    epochs=500, batch_size=256, lr=0.0005, seed=42, K=50, checkpoint_dir="checkpoints"
)
DATA_CFG = DataConfig(path="data/ihdp", replication=1, train_ratio=0.7, test_ratio=0.15)

In [20]:
RUNS = [
    ("quad", 100, 0.5, "checkpoints/final_model_naive_full_2026-08-03T16_08_35.pth"),
    ("quad", 100, 0.2, "checkpoints/final_model_naive_full_2026-08-03T16_14_47.pth"),
    ("quad", 150, 0.15, "checkpoints/final_model_naive_full_2026-08-03T16_19_01.pth"),
    ("quad", 200, 0.1, "checkpoints/final_model_naive_full_2026-08-03T16_21_12.pth"),
    ("cosine", 100, None, "checkpoints/final_model_naive_full_rep1_2026-08-16T16_51_46.pth"),
]

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds, val_ds, test_ds, ytrain_std = load_ihdp(
    DATA_CFG.path,
    replication=DATA_CFG.replication,
    train_ratio=DATA_CFG.train_ratio,
    test_ratio=DATA_CFG.test_ratio,
)
val_loader = DataLoader(val_ds, batch_size=TRAIN_CFG.batch_size)
test_loader = DataLoader(test_ds, batch_size=TRAIN_CFG.batch_size)

y_both = _DiffusionBase._assemble_yboth(train_ds.a, train_ds.y, train_ds.y_cf)
clip_value = 2 * y_both.abs().max().item()
print(f"y_std={ytrain_std:.4f}  clip_value={clip_value:.4f}\n")

header = (
    f"{'config':<22} {'rmse_y0':>9} {'rmse_y1':>9} {'pehe':>9} "
    f"{'width95_y0':>11} {'width95_y1':>11} {'cov95_y0':>9} {'cov95_y1':>9}"
)

results = {}

y_std=2.4394  clip_value=6.0188



In [ ]:
def evaluate_run(schedule, num_steps, beta_end, ckpt_path, clip_value=None):
    diff_cfg = DiffusionConfig(
        num_steps=num_steps,
        beta_start=0.0001,
        beta_end=beta_end,
        schedule=schedule,
        embedding_dim=32,
        block_dim=32,
        hidden_dim=32,
        num_blocks=4,
    )
    model = DiffPO(VAE_CFG, diff_cfg).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    torch.manual_seed(TRAIN_CFG.seed)

    result_val = evaluate(
        model,
        val_loader,
        TRAIN_CFG.K,
        device,
        sigma=1.0 / val_ds.y_std,
        clip_val=clip_value,
    )
    for k in (
        "wasserstein_y0",
        "wasserstein_y1",
        "width_95_y0",
        "width_95_y1",
        "width_99_y0",
        "width_99_y1",
        "rmse_y0",
        "rmse_y1",
        "pehe",
    ):
        result_val[k] *= ytrain_std

    return result_val

In [ ]:
macro_wd_results = []

for schedule, num_steps, beta_end, ckpt_path in RUNS:
    result = evaluate_run(schedule, num_steps, beta_end, ckpt_path)

    name = f"Schedule={schedule}, L={num_steps}"
    if beta_end is not None:
        name += f", beta_end={beta_end:.2f}"

    wd0 = result["wasserstein_y0"]
    wd1 = result["wasserstein_y1"]
    macro_wd = (wd0 + wd1) / 2
    macro_wd_results.append(macro_wd)

    print("\n" + name)
    pprint(result)
    print(f"Macro-averaged Wasserstein: {macro_wd:.4f}")

Wasserstein y0, Wasserstein y1, Macro-averaged Wasserstein

Schedule=quad, L=100, beta_end=0.50
{'coverage_95_y0': 1.0,
 'coverage_95_y1': 1.0,
 'coverage_99_y0': 1.0,
 'coverage_99_y1': 1.0,
 'pehe': 26524.608595681144,
 'rmse_y0': 23064.990762900095,
 'rmse_y1': 10052.756948044756,
 'wasserstein_y0': 47026.41437378587,
 'wasserstein_y1': 47219.33036570674,
 'width_95_y0': 204946.63878855854,
 'width_95_y1': 218102.2154898271,
 'width_99_y0': 239508.70907044038,
 'width_99_y1': 258899.00216100365}
Macro-averaged Wasserstein: 47122.8724

Schedule=quad, L=100, beta_end=0.20
{'coverage_95_y0': 1.0,
 'coverage_95_y1': 1.0,
 'coverage_99_y0': 1.0,
 'coverage_99_y1': 1.0,
 'pehe': 44.10818417948303,
 'rmse_y0': 29.868815620154464,
 'rmse_y1': 21.139186279609476,
 'wasserstein_y0': 64.17524170327108,
 'wasserstein_y1': 76.88959122729416,
 'width_95_y0': 293.0466015001184,
 'width_95_y1': 356.20532630150046,
 'width_99_y0': 349.85331913623304,
 'width_99_y1': 419.9955147753353}
Macro-averaged

In [ ]:
import numpy as np

min_idx = np.argmin(macro_wd_results)
best_run = RUNS[min_idx]
print(
    f"Best run: {best_run} with macro-averaged Wasserstein distance:"
    f" {macro_wd_results[min_idx]:.4f}\n"
)

result_with_clip = evaluate_run(*best_run, clip_value=clip_value)

wd0 = result_with_clip["wasserstein_y0"]
wd1 = result_with_clip["wasserstein_y1"]
macro_wd = (wd0 + wd1) / 2

print(result_with_clip)
print(f"Macro-averaged Wasserstein distance with clipping: {macro_wd:.4f}")

Best run: ('quad', 200, 0.1, 'checkpoints/final_model_naive_full_2026-08-03T16_21_12.pth')

{'wasserstein_y0': 6.376997970324621, 'wasserstein_y1': 6.8405913454955725, 'coverage_95_y0': 1.0, 'coverage_95_y1': 1.0, 'width_95_y0': 21.6686954615343, 'width_95_y1': 28.992124714779948, 'coverage_99_y0': 1.0, 'coverage_99_y1': 1.0, 'width_99_y0': 24.111266923104267, 'width_99_y1': 29.25617674890077, 'rmse_y0': 3.230606149149395, 'rmse_y1': 1.5137648547912192, 'pehe': 3.0029707534332886}
Macro-averaged Wasserstein distance with clipping: 6.6088
